# Unifying Bank Transactions

Combines checking-account and credit-card transactions with customer profiles into one dataset, `Unified_Transactions.csv`, which feeds the Power BI dashboard (`Unified_Transactions.pbit`).

**Inputs**
- `Customer_Info.csv`: one row per customer (demographics, income, account type)
- `Checking_Account.csv`: checking transactions (withdrawals, deposits, transfers)
- `Credit_Card.csv`: credit-card purchases (merchant, category)

In [1]:
import pandas as pd

customer_df = pd.read_csv("Customer_Info.csv")
checking_df = pd.read_csv("Checking_Account.csv")
credit_df = pd.read_csv("Credit_Card.csv")

for name, d in [("customers", customer_df), ("checking", checking_df), ("credit card", credit_df)]:
    print(f"{name:12s} {d.shape[0]:>5,} rows  {d.shape[1]} columns")

customers    1,000 rows  10 columns
checking     1,000 rows  8 columns
credit card  1,000 rows  7 columns


## 1. Clean

Parse dates and drop duplicate records so each customer and transaction appears once.

In [2]:
checking_df["Transaction_Date"] = pd.to_datetime(checking_df["Transaction_Date"])
credit_df["Transaction_Date"] = pd.to_datetime(credit_df["Transaction_Date"])
customer_df["Join_Date"] = pd.to_datetime(customer_df["Join_Date"])

customer_df = customer_df.drop_duplicates(subset="Customer_ID")
checking_df = checking_df.drop_duplicates(subset="Transaction_ID")
credit_df = credit_df.drop_duplicates(subset="Card_Id")

## 2. Align the two transaction sources

Both files number their rows from 1, so their IDs would collide once combined. Checking transactions keep their IDs (1–1,000) and credit-card transactions are offset by 1,000 (1,001–2,000), so every row in the unified table has a unique `Transaction_ID`.

Column names are aligned so the two sources stack cleanly, and a `Source` column records where each row came from.

In [3]:
checking_df["Source"] = "Checking"
credit_df["Source"] = "Credit Card"

checking_df = checking_df.rename(columns={
    "Transaction_Date": "Date",
    "Transaction_Type": "Category",
    "Branch_City": "City",
})

credit_df["Transaction_ID"] = credit_df["Card_Id"] + checking_df["Transaction_ID"].max()
credit_df = credit_df.rename(columns={"Transaction_Date": "Date"})

## 3. Combine and enrich

Stack the two sources, then join each transaction to its customer's profile. A left join keeps every transaction even if a profile were missing.

In [4]:
transactions_df = pd.concat([checking_df, credit_df], ignore_index=True)
unified_df = transactions_df.merge(customer_df, on="Customer_ID", how="left")

# Column order and names are kept stable because the Power BI template reads them by name
column_order = ["Transaction_ID", "Customer_ID", "Date", "Category", "Amount", "Account_Balance", "City_x",
                "Channel", "Source", "Card_Id", "Merchant", "First_Name", "Last_Name", "Email", "Gender",
                "City_y", "Join_Date", "Age", "Income", "Account_Type"]
unified_df = unified_df[column_order]

## 4. Validate and export

In [5]:
assert unified_df["Transaction_ID"].is_unique, "Transaction IDs must be unique"
assert unified_df["Transaction_ID"].notna().all(), "Every transaction needs an ID"
assert len(unified_df) == len(checking_df) + len(credit_df)

print(f"{len(unified_df):,} transactions from {unified_df['Customer_ID'].nunique():,} customers")
print(unified_df["Source"].value_counts().to_string())

unified_df.to_csv("Unified_Transactions.csv", index=False)
unified_df.head()

2,000 transactions from 1,000 customers
Source
Checking       1000
Credit Card    1000


,Transaction_ID,Customer_ID,Date,Category,Amount,Account_Balance,City_x,Channel,Source,Card_Id,Merchant,First_Name,Last_Name,Email,Gender,City_y,Join_Date,Age,Income,Account_Type
0,1,1,2022-03-15,Withdrawal,4057,NaN,Berovo,Online,Checking,NaN,NaN,Phyllida,Gowing,pgowing0@theglobeandmail.com,Female,Cihurip Satu,2017-01-08,38,45399,Checking
1,2,2,2024-09-06,Transfer,3145,NaN,Södertälje,Branch,Checking,NaN,NaN,Chrisy,Hulles,chulles1@phpbb.com,Male,Stylída,2021-03-01,67,148990,Both
2,3,3,2023-02-08,Transfer,3823,NaN,Shilibu,ATM,Checking,NaN,NaN,Elvera,Swannack,eswannack2@cnn.com,Female,Mianhu,2021-10-03,47,59490,Both
3,4,4,2023-04-23,Transfer,1912,NaN,Xiadong,Branch,Checking,NaN,NaN,Layton,O'Mailey,lomailey3@google.nl,Male,Krivaja,2019-11-02,39,133955,Savings
4,5,5,2023-02-04,Deposit,633,NaN,Bayan Hure,Branch,Checking,NaN,NaN,Filippa,Godson,fgodson4@mail.ru,Female,Kaiama,2024-01-11,28,119523,Both


## Column guide

| Column | Description |
|---|---|
| `City_x` | City where the transaction happened (branch or merchant city) |
| `City_y` | Customer's home city from their profile |
| `Card_Id` | Original credit-card record ID (empty for checking rows) |
| `Account_Balance`, `Channel` | Checking transactions only |
| `Merchant` | Credit-card transactions only |